# Fixation mRNN Hidden-Unit Sweep

This notebook tests how many Elman hidden units per region are needed to fit fixation-aligned neural activity PCs. The decisions from the smoke-test notebook are fixed here:

- Input is only the 3 fixation-condition one-hot channels.
- There are no Gaussian temporal basis channels.
- The target is the shared 42-dimensional PC representation of normalized firing rates for each region.
- Training runs for 25,000 iterations.
- First- and second-derivative loss weights are both 1.

The sweep trains 20, 30, 40, and 50 hidden units per region, then compares reconstruction quality, temporal-dynamics quality, and parameter-adjusted criteria.

## 1. Setup

In [ ]:
from pathlib import Path
from dataclasses import replace
from io import BytesIO

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Image, display

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = next(parent for parent in Path.cwd().parents if (parent / "src").exists())

import sys
src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from dal_monte_2022_analysis.ephys.modeling import (
    load_fixation_mrnn_config,
    make_targets,
    reconstruction_accuracy,
    replay_fixation_mrnn_run,
    settings_from_config,
    train_fixation_mrnn_scratch,
    variance_comparison,
)
from dal_monte_2022_analysis.ephys.plotting import FixationMRNNDiagnosticPlotSettings


def display_figure(fig, *, dpi=95):
    buffer = BytesIO()
    fig.savefig(buffer, format="png", dpi=dpi, bbox_inches="tight")
    display(Image(data=buffer.getvalue()))


CONDITION_COLORS = {
    "face_interactive": "#b64198",
    "face_non_interactive": "#4c9a2a",
    "object": "#6f4e37",
}


## 2. Sweep Settings

In [ ]:
cfg = load_fixation_mrnn_config(repo_root / "configs" / "ephys_fixation_mrnn.yaml")
base_settings = settings_from_config(cfg)
base_settings.dataset_cfg_path = str(repo_root / "configs" / "dataset.yaml")
base_settings.device = "auto"
base_settings.target_mode = "region_pcs"
base_settings.temporal_basis_count = 0
base_settings.epochs = 10_000
base_settings.lr = 1e-3
base_settings.temporal_derivative_loss_scale = 1.0
base_settings.temporal_curvature_loss_scale = 1.0
base_settings.l1_weight_scale = 0.01
base_settings.l1_rate_scale = 0.1
base_settings.initialization_mode = "single"
base_settings.overwrite_seed_plan = False
base_settings.train_initial_state = True

hidden_unit_grid = [30, 40, 50, 60]
base_seed = 345678
scratch_prefix = "hidden_unit_sweep_region_pcs_condition_only"

base_settings


## 3. Target Audit

This cell verifies the two key assumptions before training: the input has only the 3 condition channels, and the region-PC target has 42 PCs per region. If the PC count changes, stop here and decide whether to change the target builder or the variance threshold.

In [ ]:
targets = make_targets(base_settings)
pc_dims = {region: targets.pcs_by_region[region].shape[-1] for region in targets.region_order}
summary_rows = []
for region in targets.region_order:
    pca = targets.pca_by_region[region]
    summary_rows.append({
        "region": region,
        "pc_target_shape": targets.pcs_by_region[region].shape,
        "shared_pc_dims": pc_dims[region],
        "pcs_required_for_95pct": pca.n_components_required,
        "explained_variance_in_saved_pcs": float(np.sum(pca.explained_variance_ratio)),
        "target_variance": float(np.var(targets.pcs_by_region[region])),
    })

print("Input tensor shape:", targets.input_tensor.shape)
print("Input channels: 3 condition one-hot +", targets.input_tensor.shape[-1] - 3, "temporal channels")
display(pd.DataFrame(summary_rows))

assert targets.input_tensor.shape[-1] == 3, "Expected condition-only input with exactly 3 channels."
assert len(set(pc_dims.values())) == 1, f"Expected shared PC count across regions, got {pc_dims}."
assert next(iter(pc_dims.values())) == 42, f"Expected 42 PCs per region, got {pc_dims}."


## 4. Fit-Quality Metrics

For model selection, use several metrics rather than a single scalar:

- Pointwise reconstruction loss: direct PC value fit.
- First- and second-derivative losses: fit of temporal slope and curvature across adjacent 10 ms bins.
- Mean R2 and correlation: interpretable reconstruction summaries by region and fixation condition.
- Variance ratio: whether reconstructed trajectories recover target variance rather than collapsing to flat outputs.
- AIC/BIC-style scores: useful as a parameter-count penalty, but heuristic here because the loss includes derivative terms and observations are temporally correlated.

AIC/BIC are computed from pointwise residual SSE under a Gaussian iid approximation:

$$AIC = 2k + n\log(SSE/n)$$

$$BIC = k\log(n) + n\log(SSE/n)$$

where $k$ is the number of trainable parameters and $n$ is the number of scalar PC target observations.

In [ ]:
def plot_loss(history, title):
    fig, ax = plt.subplots(figsize=(6.8, 3.4), dpi=130)
    for column, label in [
        ("loss", "total"),
        ("reconstruction_loss", "pointwise"),
        ("temporal_derivative_loss", "first derivative"),
        ("temporal_curvature_loss", "second derivative"),
        ("rate_loss", "L1 rate"),
        ("weight_loss", "L1 weight"),
    ]:
        if column in history:
            final_value = float(history[column].iloc[-1])
            ax.plot(history["iteration"], history[column], label=f"{label}: {final_value:.4g}")
    ax.set_title(title)
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Loss")
    ax.grid(alpha=0.25)
    ax.legend(frameon=False, fontsize=8)
    display_figure(fig)
    return fig, ax


def train_hidden_unit_model(hidden_units):
    run_settings = replace(
        base_settings,
        hidden_units=int(hidden_units),
        seed=base_seed + int(hidden_units),
    )
    scratch_id = f"{scratch_prefix}_h{hidden_units:03d}"
    result = train_fixation_mrnn_scratch(run_settings, scratch_id=scratch_id, overwrite=True)
    replay = replay_fixation_mrnn_run(result["run_dir"], device="cpu")
    print(f"hidden_units={hidden_units}; input_shape={replay['checkpoint']['input_tensor'].shape}; run_dir={result['run_dir']}")
    return {"hidden_units": int(hidden_units), "settings": run_settings, "result": result, "replay": replay}


def count_trainable_parameters(replay):
    model = replay["model"]
    n_model = sum(param.numel() for param in model.parameters() if param.requires_grad)
    h0 = replay["checkpoint"].get("h0")
    n_h0 = int(np.prod(tuple(h0.shape))) if h0 is not None else 0
    return int(n_model + n_h0)


def pointwise_sse_and_n(replay):
    sse = 0.0
    n = 0
    for region in replay["region_order"]:
        observed = np.asarray(replay["checkpoint"]["target_by_region"][region], dtype=float)
        predicted = replay["output_by_region"][region].detach().cpu().numpy().astype(float, copy=False)
        residual = observed - predicted
        sse += float(np.sum(residual ** 2))
        n += int(residual.size)
    return sse, n


def derivative_sse_and_n(replay, *, order):
    sse = 0.0
    n = 0
    for region in replay["region_order"]:
        observed = np.asarray(replay["checkpoint"]["target_by_region"][region], dtype=float)
        predicted = replay["output_by_region"][region].detach().cpu().numpy().astype(float, copy=False)
        for _ in range(int(order)):
            observed = np.diff(observed, axis=1)
            predicted = np.diff(predicted, axis=1)
        residual = observed - predicted
        sse += float(np.sum(residual ** 2))
        n += int(residual.size)
    return sse, n


def model_summary_row(fit):
    replay = fit["replay"]
    history = fit["result"]["history"]
    accuracy = reconstruction_accuracy(replay)
    variance = variance_comparison(replay)
    sse, n_obs = pointwise_sse_and_n(replay)
    dsse, dn_obs = derivative_sse_and_n(replay, order=1)
    csse, cn_obs = derivative_sse_and_n(replay, order=2)
    k = count_trainable_parameters(replay)
    sigma2 = max(sse / max(n_obs, 1), np.finfo(float).tiny)
    return {
        "hidden_units_per_region": fit["hidden_units"],
        "total_hidden_units": int(fit["replay"]["model"].total_num_units),
        "trainable_parameters": k,
        "n_scalar_observations": n_obs,
        "final_total_loss": float(history["loss"].iloc[-1]),
        "final_reconstruction_loss": float(history["reconstruction_loss"].iloc[-1]),
        "final_first_derivative_loss": float(history["temporal_derivative_loss"].iloc[-1]),
        "final_second_derivative_loss": float(history["temporal_curvature_loss"].iloc[-1]),
        "pointwise_sse": sse,
        "first_derivative_sse": dsse,
        "second_derivative_sse": csse,
        "mean_r2": float(accuracy["r2"].mean()),
        "mean_correlation": float(accuracy["correlation"].mean()),
        "mean_variance_ratio": float(variance["reconstructed_to_observed_ratio"].mean()),
        "aic_pointwise": float(2 * k + n_obs * np.log(sigma2)),
        "bic_pointwise": float(k * np.log(max(n_obs, 1)) + n_obs * np.log(sigma2)),
    }


## 5. Train Hidden-Unit Sweep

This cell trains four models for 25,000 iterations each. Rerunning overwrites the scratch checkpoints for the same hidden-unit counts.

In [ ]:
fits = []
for hidden_units in hidden_unit_grid:
    fit = train_hidden_unit_model(hidden_units)
    fits.append(fit)
    plot_loss(fit["result"]["history"], f"Hidden Units per Region = {hidden_units}")


## 6. Compare Fit Quality Across Hidden-Unit Counts

In [ ]:
summary = pd.DataFrame([model_summary_row(fit) for fit in fits])
summary = summary.sort_values("hidden_units_per_region").reset_index(drop=True)
display(summary)

metric_columns = [
    "final_total_loss",
    "final_reconstruction_loss",
    "final_first_derivative_loss",
    "final_second_derivative_loss",
    "mean_r2",
    "mean_correlation",
    "mean_variance_ratio",
    "aic_pointwise",
    "bic_pointwise",
]
fig, axes = plt.subplots(3, 3, figsize=(12, 9), dpi=130)
for ax, metric in zip(axes.flat, metric_columns):
    ax.plot(summary["hidden_units_per_region"], summary[metric], marker="o", linewidth=1.6)
    ax.set_title(metric)
    ax.set_xlabel("Hidden units per region")
    ax.grid(alpha=0.25)
for ax in axes.flat[len(metric_columns):]:
    ax.axis("off")
fig.tight_layout()
display_figure(fig)


## 7. Top-PC Reconstruction Checks

These plots show observed and reconstructed trajectories for the top three PC targets in each region and fixation type. Observed traces are solid; mRNN reconstructions are dotted in the same condition color. Use these plots to check whether lower-dimensional models miss transient dynamics even when aggregate metrics are acceptable.

In [ ]:
def plot_top_pc_reconstructions_by_hidden(fits, *, n_pcs=3):
    conditions = tuple(fits[0]["replay"]["condition_order"])
    regions = tuple(fits[0]["replay"]["region_order"])
    time = np.asarray(fits[0]["replay"]["checkpoint"]["timeline_s"], dtype=float)
    for region in regions:
        fig, axes = plt.subplots(
            len(fits),
            int(n_pcs),
            figsize=(4.2 * int(n_pcs), 2.1 * len(fits)),
            dpi=130,
            sharex=True,
            squeeze=False,
        )
        for row, fit in enumerate(fits):
            replay = fit["replay"]
            observed = np.asarray(replay["checkpoint"]["target_by_region"][region], dtype=float)
            predicted = replay["output_by_region"][region].detach().cpu().numpy()
            for pc_idx in range(int(n_pcs)):
                ax = axes[row, pc_idx]
                for cond_idx, condition in enumerate(conditions):
                    color = CONDITION_COLORS.get(condition)
                    ax.plot(time, observed[cond_idx, :, pc_idx], color=color, linewidth=1.5, linestyle="-", label=f"{condition} observed")
                    ax.plot(time, predicted[cond_idx, :, pc_idx], color=color, linewidth=1.8, linestyle=":", label=f"{condition} mRNN")
                if row == 0:
                    ax.set_title(f"{region} PC{pc_idx + 1}")
                if pc_idx == 0:
                    ax.set_ylabel(f"h={fit['hidden_units']}\\nscore")
                if row == len(fits) - 1:
                    ax.set_xlabel("Time (s)")
                ax.grid(alpha=0.2)
        handles, labels = axes[0, 0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False, fontsize=7)
        fig.suptitle(f"{region}: Top PC Reconstructions Across Hidden-Unit Counts", y=1.02)
        fig.tight_layout()
        display_figure(fig)


plot_top_pc_reconstructions_by_hidden(fits, n_pcs=3)


## 8. Region- and Condition-Level Diagnostics

Aggregate metrics can hide region-specific failures. This section breaks reconstruction quality down by hidden-unit count, region, and fixation condition.

In [ ]:
accuracy_rows = []
variance_rows = []
for fit in fits:
    h = fit["hidden_units"]
    acc = reconstruction_accuracy(fit["replay"]).assign(hidden_units_per_region=h)
    var = variance_comparison(fit["replay"]).assign(hidden_units_per_region=h)
    accuracy_rows.append(acc)
    variance_rows.append(var)
accuracy_by_fit = pd.concat(accuracy_rows, ignore_index=True)
variance_by_fit = pd.concat(variance_rows, ignore_index=True)

display(accuracy_by_fit)
display(variance_by_fit)

for metric in ["r2", "correlation", "mse"]:
    pivot = accuracy_by_fit.pivot_table(
        index=["region", "condition"],
        columns="hidden_units_per_region",
        values=metric,
    )
    print(metric)
    display(pivot)


## 9. Other Losses to Consider

The current objective already includes pointwise PC reconstruction, first derivative, and second derivative losses. Additional losses worth testing:

- Region-balanced loss: average each region's loss before summing, so regions with larger variance or harder PCs do not dominate.
- PC-weighted loss: weight PCs by explained variance if the top PCs should dominate, or inversely by variance if small PCs should be preserved.
- Variance-matching loss: penalize mismatch between reconstructed and observed variance within each region/condition, reducing flat-output solutions.
- Correlation loss: maximize temporal correlation between observed and reconstructed PC trajectories, useful when shape matters more than absolute scale.
- Spectral or frequency-domain loss: compare FFT power in low/mid/high temporal bands, encouraging the RNN to capture faster temporal variations.
- Peak/trough timing loss: penalize shifts in extrema or onset response peaks, if specific fixation-locked events are scientifically important.
- Autonomy penalty or input-ablation consistency: if we want less input-driven behavior, add diagnostics/losses that test whether hidden dynamics can continue trajectories after a brief cue.
- Mild L2 weight decay: controls parameter growth more smoothly than L1 sparsity and can help compare larger hidden-unit models.

For this hidden-unit sweep, AIC/BIC-style penalties plus visual top-PC checks should help decide whether extra hidden units buy real temporal reconstruction or only overparameterization.